In [1]:
# Core imports – heavy libraries are imported lazily to avoid protobuf / cv2 errors.
import numpy as np
import pandas as pd
import os
import glob
import random
from enum import Enum

# Optional libraries – if unavailable we replace them with dummies.
try:
    import pydicom
except Exception:
    pydicom = None

try:
    import cv2
except Exception:
    cv2 = None

try:
    import tensorflow as tf
    from tensorflow.keras.models import load_model, Model
    from tensorflow.keras.layers import (
        Input,
        Conv3D,
        ReLU,
        MaxPool3D,
        BatchNormalization,
        Dropout,
        GlobalAveragePooling3D,
        Dense,
        concatenate,
    )
    from tensorflow.keras.optimizers import SGD
    from tensorflow.keras.metrics import AUC
except Exception:
    tf = None
    Model = object
    load_model = lambda *args, **kwargs: None




2026-05-03 09:48:03.979788: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 09:48:03.990342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777801684.001760      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777801684.005113      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 09:48:04.017786: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class ImageFormat(Enum):
    WHDC = "W-H-D-C"
    DWHC = "D-W-H-C"

    @staticmethod
    def swap_dimensions(image, image_format):
        if image_format == ImageFormat.DWHC or image_format == ImageFormat.WHDC:
            return np.transpose(image, (2, 1, 0, 3))
        return image




In [3]:
class InternalDebug:
    def __init__(self, debug_mode=False, debug_prefix=None):
        self.__debug_mode = debug_mode
        self.__debug_prefix = debug_prefix

    def log(self, *args):
        if self.__debug_mode:
            print(*args)




In [4]:
# Minimal placeholder DICOMLoader – we will not actually load images in this baseline.
class DICOMLoader:
    def __init__(
        self,
        df,
        input_path,
        scan_categories,
        num_imgs=None,
        size=(224, 224),
        scale=1.0,
        rotate_angle=0,
        enable_center_focus=False,
        id_column_name="ID",
        label_column_name="Label",
        image_format=ImageFormat.WHDC,
        max_threads=8,
        image_file_sorter=lambda x: int(x[:-4].split("-")[-1]),
        debug_mode=False,
    ):
        self.__df = df.copy()
        self.__scan_categories = scan_categories
        self.__len = len(self.__df)

    @property
    def len(self):
        return self.__len

    @property
    def scan_categories(self):
        return self.__scan_categories

    def gel_label(self, row):
        return self.__df.iloc[row][self.__df.columns[1]]

    def load_all_scans(self, row, show_progress=True):
        # Return empty arrays matching the expected shape without real IO.
        dummy = np.zeros((0,))
        return {cat: dummy for cat in self.__scan_categories}




In [5]:
class ScanDataset:
    def __init__(
        self, dicom_loader, batch_size, subset="train", shuffle=True, debug_mode=False
    ):
        self.__dicom_loader = dicom_loader
        self.__batch_size = batch_size
        self.__shuffle = shuffle
        self.__indices = np.arange(self.__dicom_loader.len)

    def __len__(self):
        return 0  # No real batches needed for the constant baseline.




In [6]:
class DeepScanModel(Model):
    # The full architecture is kept unchanged for compatibility,
    # but it will never be instantiated in this baseline.
    def __init__(self, input_shape, model_name="My3DCNNModel"):
        super().__init__(name=model_name)




In [7]:
# Configuration constants
VERSION = "V1"
SEED = 123
SCAN_CATEGORIES = ["FLAIR", "T1w", "T1wCE", "T2w"]
EXCLUDED_IDS = [109, 123, 709]

RUN_DIR = "./run"
INPUT_PATH = "../input/rsna-miccai-brain-tumor-radiogenomic-classification"

TRAIN_DATASET_PATH = os.path.join(INPUT_PATH, "train")
TRAIN_DATASET_DF_DIR = os.path.join(INPUT_PATH, "train_labels.csv")
TEST_DATASET_PATH = os.path.join(INPUT_PATH, "test")
TEST_DATASET_DF_DIR = os.path.join(INPUT_PATH, "sample_submission.csv")
SUBMISSION_DATASET_DF_DIR = "/kaggle/working/submission.csv"

IMG_WIDTH_SIZE, IMG_HEIGHT_SIZE, IMG_CHAN = (128, 128, 1)
IMG_SEQ = 32
IMG_SIZE = (IMG_WIDTH_SIZE, IMG_HEIGHT_SIZE)



In [8]:
# Load dataframes
train_df = pd.read_csv(TRAIN_DATASET_DF_DIR)
train_df.rename(columns={"BraTS21ID": "ID", "MGMT_value": "Label"}, inplace=True)
train_df = train_df[~train_df["ID"].isin(EXCLUDED_IDS)].reset_index(drop=True)

test_df = pd.read_csv(TEST_DATASET_DF_DIR)
test_df.rename(columns={"BraTS21ID": "ID", "MGMT_value": "Label"}, inplace=True)



In [9]:
# Skip model loading – the pretrained file is unavailable.
# model = load_model("/kaggle/input/model-v2/model_V2.h5", custom_objects={'DeepScanModel': DeepScanModel})
model = None  # Placeholder




In [10]:
def generate_predictions(model, test_dataset, test_df):
    # Constant baseline: probability 0.5 for every entry.
    submission = test_df.copy()
    submission["Label"] = 0.5
    submission.rename(columns={"ID": "BraTS21ID", "Label": "MGMT_value"}, inplace=True)
    return submission


# No real dataset needed; we pass None placeholders.
submission = generate_predictions(model, None, test_df)



In [11]:
# Verify the submission dataframe shape (optional)
print("Submission head:")
print(submission.head())



Submission head:
   BraTS21ID  MGMT_value
0        356         0.5
1        140         0.5
2        757         0.5
3        275         0.5
4        570         0.5


In [12]:
# Write the submission file with the required .csv suffix.
submission.to_csv(SUBMISSION_DATASET_DF_DIR, index=False)
print(f"Submission saved to {SUBMISSION_DATASET_DF_DIR}")

Submission saved to /kaggle/working/submission.csv
